# Spectrum Analysis Example

This notebook demonstrates a complete gamma-ray spectrum analysis workflow using
the `gs_analysis` and `gs_spe_reading` modules.

The example uses a real measurement of an ATR (research reactor) sample
(`atr_sample_1.Spe`) from the `test_data/` directory. The workflow covers:

1. **Reading** a `.Spe` file (Maestro ASCII format) and generating the energy axis.
2. **Plotting** the raw spectrum.
3. **Finding peaks** using the scipy-prominence method.
4. **Calculating net counts** for each detected peak with trapezoid background subtraction.
5. **Accessing metadata** such as real time from the spectrum object.

## 1. Setup – import modules

In [ ]:
import sys
import os

# Add the package root to sys.path when running from the examples/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

import gs_analysis as ga
import gs_spe_reading

## 2. Load the spectrum

Read the `.Spe` file using `gs_spe_reading.read_dollar_spe`, then generate the
energy-bin axis from the calibration coefficients stored in the file using
`ga.generate_ebins`.

In [ ]:
# Path to the ATR sample spectrum relative to the examples/ directory
SPE_PATH = os.path.join('..', 'test_data', 'atr_sample_1.Spe')

spec = gs_spe_reading.read_dollar_spe(SPE_PATH)
spec.ebins = ga.generate_ebins(spec)  # keV per channel

print(f"Channels    : {spec.num_channels}")
print(f"Real time   : {spec.real_time:.1f} s")
print(f"Total counts: {spec.counts.sum():,}")
print(f"Energy range: {spec.ebins[0]:.1f} - {spec.ebins[-1]:.1f} keV")

## 3. Plot the raw spectrum

`ga.plot_spec` provides a quick way to visualise counts against energy. The ATR
sample shows multiple activation product peaks spread across the spectrum.

In [ ]:
ga.plot_spec(spec.counts, erg=spec.ebins)

## 4. Find peaks

`ga.peak_finder` applies 5-point smoothing and then uses
`scipy.signal.find_peaks` with a prominence threshold to locate photopeaks.

- **`prominence`** – minimum vertical distance between a peak and its surrounding
  baseline. Increase to suppress weak peaks; decrease to detect more peaks.
- **`wlen`** – window length (channels) used when computing prominence.

In [ ]:
# Tune these parameters for your detector and source
PROMINENCE = 100  # minimum peak prominence in counts
WLEN       = 10   # window length in channels

smoothed_counts, peaks = ga.peak_finder(spec.counts, PROMINENCE, WLEN)

print(f"Peaks found (channel index): {peaks}")
print(f"Corresponding energies (keV): {spec.ebins[peaks].round(1)}")

## 5. Calculate net counts for each peak

`ga.peak_counts` combines ROI extraction and trapezoid background subtraction into
a single call. It returns the channel index of the selected peak and the estimated
net counts (gross counts minus background).

Iterating over all detected peaks gives a summary table of energies and net counts.

In [ ]:
net_counts_arr = []

print(f"{'Peak #':<8} {'Channel':<10} {'Energy (keV)':<15} {'Net counts':<12}")
print("-" * 47)

for i in range(len(peaks)):
    peak_index, net = ga.peak_counts(peaks, i, smoothed_counts, spec.ebins)
    net_counts_arr.append(net)
    print(f"{i:<8} {peak_index:<10} {spec.ebins[peak_index]:<15.1f} {net:<12.0f}")

net_counts_arr = np.array(net_counts_arr)

## 6. Access spectrum metadata

The spectrum object exposes acquisition metadata such as real time and live time
that can be used to compute count rates or correct for dead-time losses.

In [ ]:
real_time = spec.real_time
print(f"Real time : {real_time:.1f} s")

# Example: compute net count rate for each peak
print("\nNet count rates (counts per second):")
for i, (ch, net) in enumerate(zip(peaks, net_counts_arr)):
    print(f"  Peak {i}: {spec.ebins[ch]:.1f} keV  ->  {net / real_time:.2f} cps")

## Summary

This notebook demonstrated a basic gamma-ray spectrum analysis workflow:

1. **Load** a measured spectrum from a `.Spe` file and generate its energy axis.
2. **Plot** the spectrum to inspect its overall shape.
3. **Detect peaks** using the scipy-prominence method via `ga.peak_finder`.
4. **Quantify** each peak by computing background-subtracted net counts with
   `ga.peak_counts`.
5. **Use metadata** (real time) to convert counts to count rates.

For more advanced peak-finding options, including the Mariscotti second-difference
method, see the `peak_find_example.ipynb` notebook in this directory.